In [1]:
import csv
import json
import os
import time
import pandas as pd
import numpy as np
import cv2

import torch
from accelerate import Accelerator
from torch.utils.tensorboard import SummaryWriter



from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.utils import set_determinism

/home/ubuntu/miniconda3/envs/segmentation/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
<frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


In [2]:
from monai.transforms import (
    AsDiscrete,
    AsDiscreted,
    EnsureChannelFirstd,
    EnsureTyped,
    Compose,
    CropForegroundd,
    LoadImaged,
    Orientationd,
    RandCropd,
    RandSpatialCropd,
    RandCropByPosNegLabeld,
    SaveImaged,
    ScaleIntensityRanged,
    ScaleIntensityd,
    Spacingd,
    Invertd,
    MapTransform,
    SpatialPadd,
)
from monai.data import CacheDataset, DataLoader, Dataset

In [3]:
import matplotlib.pyplot as plt

In [4]:
set_determinism(42)

## Config

In [5]:
class TrainCfg:
    #data
    data_root: str = './inputs'
    train_ids: str = './inputs/splits/train_case_days.csv'
    val_ids: str = './inputs/splits/val_case_days.csv'
    num_workers: int = 2

    # training
    patch_d: int = 96
    patch_h: int = 224
    patch_w: int = 224

    # TensorBoard val sample visualization
    val_vis_every: int = 5
    val_vis_samples: str = './inputs/splits/val_vis_samples.json'

cfg = TrainCfg()

## Data Utils

In [6]:
def parse_scan_filename(filename: str) -> tuple[str, int, int]:
    """From slice_XXXX_H_W_psx_psy.png get slice index and H,W in filename"""
    parts = filename.split('_')
    slice_idx = int(parts[1])
    height = int(parts[2])
    width = int(parts[3])
    return filename, slice_idx, height, width

def build_case_day_slices(data_dir: str) -> dict[str, list[str]]:
    """Returns dict: key = 'case123_day0', value = sorted list of scan file paths."""
    out: dict[str, list[str]] = {}
    for case in sorted(os.listdir(data_dir)):
        case_path = os.path.join(data_dir, case)

        for day in sorted(os.listdir(case_path)):
            day_path = os.path.join(case_path, day)
            scans_dir = os.path.join(day_path, 'scans')

            scans_files = [os.path.join(scans_dir, f) for f in sorted(os.listdir(scans_dir)) if f.endswith('.png')]
        
            scans_files.sort(key=lambda f: parse_scan_filename(os.path.basename(f))[1])
            out[day] = scans_files
    return out

def build_rle_index(train_csv_path: str) -> dict[tuple[str, int], dict[str, str]]:
    """Map train.csv to idx[(case_day, slice_idx)] = {class_name: rle_string}."""
    df = pd.read_csv(train_csv_path)
    idx: dict[tuple[str, int], dict[str, str]] = {}

    for _, row in df.iterrows():
        _id = row["id"]
        cls = row["class"]
        seg = row["segmentation"]

        parts = _id.split("_")
        case_day = "_".join(parts[0:2])
        slice_idx = int(parts[-1])
        key = (case_day, slice_idx)
        idx.setdefault(key, {})[cls] = seg

    return idx    

def load_case_days(ids_path: str) -> list[str]: 
    """Load case_days from csv file."""
    case_days = []
    with open(ids_path, 'r') as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            case_days.append(row[0])
    return case_days

def load_val_vis_samples(path: str) -> list[dict]:
    with open(path, 'r') as f:
        samples = json.load(f)
    return samples

## Build metainfo

In [7]:
# Build slices index
data_dir = os.path.join(cfg.data_root, 'train')
data_csv = os.path.join(cfg.data_root, 'train.csv')

case_day_slices = build_case_day_slices(data_dir)
all_case_days = sorted(case_day_slices.keys())
print(f"Found case_day volumes: {len(all_case_days)}")

# Build rle index
rle_index = build_rle_index(data_csv)

train_set = set(load_case_days(cfg.train_ids))
val_set = set(load_case_days(cfg.val_ids))

overlap = sorted(train_set & val_set)
if overlap:
    raise ValueError(f"train_ids and val_ids overlap (n={len(overlap)}), e.g. {overlap[:5]}")

print(f"Train case_days: {len(train_set)}, Val case_days: {len(val_set)}")

train_files_d = [{"case_day": cd} for cd in sorted(train_set)]
val_files_d = [{"case_day": cd} for cd in sorted(val_set)]

val_vis_samples = load_val_vis_samples(cfg.val_vis_samples)
print(f"Validation visualization samples: {len(val_vis_samples['samples'])}")

Found case_day volumes: 274
Train case_days: 216, Val case_days: 58
Validation visualization samples: 16


## Build dataset

### RLE

In [8]:
def rle_decode(rle: str, height: int, width: int) -> np.ndarray:
    """Kaggle RLE decode for 2D mask."""
    if rle is None or rle == "" or (isinstance(rle, float) and np.isnan(rle)):
        return np.zeros((height, width), dtype=np.uint8)

    s = rle.strip().split()
    starts = np.asarray(s[0::2], dtype=int) - 1
    lengths = np.asarray(s[1::2], dtype=int)
    ends = starts + lengths

    img = np.zeros(height * width, dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1

    return img.reshape((height, width))


def rle_encode(img: np.ndarray) -> str:
    """Kaggle RLE encode for 2D mask (expects {0,1} uint8)."""
    pixels = img.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

### Transform

In [9]:
CLASSES = ["large_bowel", "small_bowel", "stomach"]
CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}

class LoadCaseDayVolumed(MapTransform):
    def __init__(self, keys, case_day_slices, rle_index):
        super().__init__(keys)
        self.case_day_slices = case_day_slices
        self.rle_index = rle_index

    def __call__(self, data):
        d = dict(data)
        case_day = d["case_day"]
        slice_files = self.case_day_slices[case_day]

        img0 = cv2.imread(slice_files[0], cv2.IMREAD_GRAYSCALE)
        H, W = img0.shape
        D = len(slice_files)

        img_vol = np.zeros((D, H, W), dtype=np.uint8)
        mask_vol = np.zeros((len(CLASSES), D, H, W), dtype=np.uint8)

        for z, f in enumerate(slice_files):
            slice_idx = parse_scan_filename(os.path.basename(f))[1]
            img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
            img_vol[z] = img

            rle_map = self.rle_index.get((case_day, slice_idx), {})
            for cls_name, cls_i in CLASS2IDX.items():
                m2d = rle_decode(rle_map.get(cls_name, ""), H, W)
                mask_vol[cls_i, z] = m2d

        d["image"] = img_vol
        d["label"] = mask_vol
        return d

In [10]:
train_transforms = Compose(
    [
        LoadCaseDayVolumed(
            keys=["case_day"],
            case_day_slices=case_day_slices,
            rle_index=rle_index,
        ),
        EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
        ScaleIntensityd(
            keys=["image"],
            minv=0.0,
            maxv=1.0,
        ),
        SpatialPadd(
            keys=["image", "label"],
            spatial_size=(cfg.patch_d, cfg.patch_h, cfg.patch_w),
            mode="constant",
        ),
        RandSpatialCropd(
            keys=["image", "label"],
            roi_size=(cfg.patch_d, cfg.patch_h, cfg.patch_w),
            random_size=False,
        ),
        EnsureTyped(keys=["image", "label"]),
    ]
)

val_transforms = Compose(
    [
        LoadCaseDayVolumed(
            keys=["case_day"],
            case_day_slices=case_day_slices,
            rle_index=rle_index,
        ),
        EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
        ScaleIntensityd(
            keys=["image"],
            minv=0.0,
            maxv=1.0,
        ),
        SpatialPadd(
            keys=["image", "label"],
            spatial_size=(cfg.patch_d, cfg.patch_h, cfg.patch_w),
            mode="constant",
        ),
        EnsureTyped(keys=["image", "label"]),
    ]
)

### Datasets

In [ ]:
train_ds = CacheDataset(
    data=train_files_d,
    transform=train_transforms,
    cache_rate=0.5,
    num_workers=cfg.num_workers,
    progress=True,
)

val_ds = CacheDataset(
    data=val_files_d,
    transform=val_transforms,
    cache_rate=1.0,
    num_workers=cfg.num_workers,
    progress=True,
)

print(f"Train dataset size: {len(train_ds)}")
print(f"Val dataset size: {len(val_ds)}")

Loading dataset:   0%|          | 0/216 [00:00<?, ?it/s]

Loading dataset: 100%|██████████| 58/58 [00:30<00:00,  1.88it/s]


Train dataset size: 216
Val dataset size: 58


## Model, Loss, Optimizer

In [12]:
from monai.networks.nets import Unet
from monai.networks.layers import Norm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Unet(
    spatial_dims=3,
    in_channels=1,
    out_channels=len(CLASSES),
    channels=(32, 64, 128, 256, 512),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    dropout=0.2,
    norm=Norm.BATCH,
).to(device)

loss_function = DiceCELoss(sigmoid=True, squared_pred=True, reduction="mean").to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

dice_metric = DiceMetric(
    include_background=True,
    reduction="mean_batch",
    get_not_nans=False,
)

## Training

In [ ]:
epochs = 150
val_interval = 5
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []

In [ ]:
from monai.inferers import sliding_window_inference
from monai.data import pad_list_data_collate

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
    collate_fn=pad_list_data_collate,  # 处理不同尺寸的样本
)
val_loader = DataLoader(
    val_ds,
    batch_size=1,  # 验证时使用 batch_size=1 避免 collate 问题
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

roi_size = (cfg.patch_d, cfg.patch_h, cfg.patch_w)
sw_batch_size = 16  # 48GB显存可以用更大的batch，加速推理
sw_overlap = 0.25

for epoch in range(epochs):
    print("-" * 60)
    print(f"epoch {epoch + 1}/{epochs}")

    # ---- train ----
    model.train()
    epoch_loss = 0.0
    step = 0

    for batch_data in train_loader:
        step += 1
        inputs = batch_data["image"].to(device)
        labels = batch_data["label"].to(device).float()

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=use_amp):
            logits = model(inputs)
            loss = loss_function(logits, labels)

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"  step {step:4d} | NaN/Inf loss, skipping batch")
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

        if step % 20 == 0:
            print(f"  step {step:4d} | loss {loss.item():.4f}")

    epoch_loss /= max(step, 1)
    epoch_loss_values.append(epoch_loss)
    print(f"train loss: {epoch_loss:.4f}")

    # ---- val ----
    if (epoch + 1) % val_interval == 0:
        model.eval()
        dice_metric.reset()
        val_loss = 0.0
        val_steps = 0

        with torch.no_grad():
            for val_data in val_loader:
                val_steps += 1
                val_inputs = val_data["image"].to(device)
                val_labels = val_data["label"].to(device).float()

                with torch.amp.autocast("cuda", enabled=use_amp):
                    val_logits = sliding_window_inference(
                        val_inputs,
                        roi_size=roi_size,
                        sw_batch_size=sw_batch_size,
                        predictor=model,
                        overlap=sw_overlap,
                    )
                    vloss = loss_function(val_logits, val_labels)

                val_loss += vloss.item()

                val_probs = torch.sigmoid(val_logits)
                val_preds = (val_probs > 0.5).float()

                dice_metric(y_pred=val_preds, y=val_labels)

        val_loss /= max(val_steps, 1)
        dice_per_class = dice_metric.aggregate()
        dice_overall = dice_per_class.mean().item()

        metric_values.append(dice_overall)

        print(f"val loss: {val_loss:.4f}")
        print("val dice per class:", dice_per_class.detach().cpu().numpy())
        print(f"val dice overall: {dice_overall:.4f}")

        if dice_overall > best_metric:
            best_metric = dice_overall
            best_metric_epoch = epoch + 1
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_metric": best_metric,
                "dice_per_class": dice_per_class.detach().cpu().numpy(),
            }, "best.pt")
            print("saved new best model -> best.pt")

    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_metric": best_metric,
        "best_metric_epoch": best_metric_epoch,
        "epoch_loss_values": epoch_loss_values,
        "metric_values": metric_values,
    }, "last.pt")

print(f"Training done. best_metric={best_metric:.4f} at epoch={best_metric_epoch}")

------------------------------------------------------------
epoch 1/5


  step   20 | loss 0.9982
  step   40 | loss 0.9954
  step   60 | loss 0.9950
  step   80 | loss 0.9917
  step  100 | loss 0.9891
train loss: 0.9947


/home/ubuntu/miniconda3/envs/segmentation/lib/python3.10/site-packages/monai/inferers/utils.py:226: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  win_data = torch.cat([inputs[win_slice] for win_slice in unravel_slice]).to(sw_device)
/home/ubuntu/miniconda3/envs/segmentation/lib/python3.10/site-packages/monai/inferers/utils.py:370: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorc

val loss: 1.0036
val dice per class: [0.01522496 0.02048283 0.01175351]
val dice overall: 0.0158
saved new best model -> best.pt
------------------------------------------------------------
epoch 2/5
  step   20 | loss 0.9852
  step   40 | loss 0.9817
  step   60 | loss 0.9782
  step   80 | loss 0.9692
  step  100 | loss 0.9753


KeyboardInterrupt: 

In [ ]:
# 从 last.pt 恢复训练
# ckpt = torch.load("last.pt")
# model.load_state_dict(ckpt["model_state_dict"])
# optimizer.load_state_dict(ckpt["optimizer_state_dict"])
# scaler.load_state_dict(ckpt["scaler_state_dict"])
# start_epoch = ckpt["epoch"]
# best_metric = ckpt["best_metric"]
# best_metric_epoch = ckpt["best_metric_epoch"]
# epoch_loss_values = ckpt["epoch_loss_values"]
# metric_values = ckpt["metric_values"]

# 从 best.pt 加载最佳模型用于推理
# ckpt = torch.load("best.pt")
# model.load_state_dict(ckpt["model_state_dict"])